# Tutorial -  Volatilidad
Sergio Cabrales, Universidad de los Andes

https://www.sac2.com/

## 1. Carga de librerías, funciones y APIs necesarias.

#### 1.1. Instalan las librerías que no incluye Google Colab

In [ ]:
pip install yfinance

In [ ]:
pip install mplfinance

#### 1.2. Se cargan las librerías requeridas

In [ ]:
# Funciones numéricas adicionales
import numpy as np

# Lectura de datos y manejo de Data-sets
import pandas as pd
import matplotlib.pyplot as plt


# Datos
import yfinance as yfin

# Gráficos
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

#analisis tecnico
import mplfinance as mpf

# Probabilidad y estadística
import math
from scipy.stats import norm, chi2, jarque_bera
from scipy.optimize import brentq
from scipy import stats


## 2. Obtención de datos históricos

#### 2.1. Descarga de datos desde Yahoo Finance

https://finance.yahoo.com/


In [ ]:
# Descargamos datos de la acción sleccionada:
df = yfin.download('GOOGL', start='2000-01-01', multi_level_index=False)
df

## 3. Visualización y Estadísticas Descriptivas

### 3.1. Utiliza la librería mpf para hacer un gráfico de velas japonesas de la acción

In [ ]:
mpf.plot(df,type='candle', volume=True,figratio=(19,8),style='yahoo')

## 4. Retornos

### 4.1. Retornos Logarítmicos

Los retornos logarítmicos se calculan como:
$$
r_{t} = ln \left( \frac{S_t}{S_{t-1}} \right ) = ln \left( S_{t} \right) - ln \left( S_{t-1} \right)
$$

In [ ]:
# Guardamos los retornos logaritmicos en una nueva columna.
df['Log Returns'] = np.log(df['Close']) - np.log(df['Close'].shift(1))
df['Log Returns'][0] = 0
df

### 4.3. Retornos Logarítmicos anualizados

Podemos calcular el log-retorno anual ($r$) como el número de días bursátiles (252 días) por el promedio del log-retorno diario:

$$
r = 252 \bar{r_t}
$$

In [ ]:
# Podemos imprimir el retornos anual:
LogReturns = np.mean(df["Log Returns"])*252
LogReturns

### 4.4. Gráfica de retornos
- Podemos graficar los retornos igual que como graficamos los precios.

In [ ]:
# Gráfico de los retornos logarítmicos
plt.figure(figsize=(15,8))
plt.plot(df['Log Returns'], color = 'red')
plt.title('Retornos Logarítmicos')
plt.xlabel('Fecha')
plt.show()

## 5. Volatilidad

### 5.1 Volatilidad diaria y anual

La volatilidad diaria del activo es la desviación estándar de sus retornos o la raíz de la varianza:  

$$vol=desv(r)=\sqrt{Var(r)}$$

En finanzas, se utiliza con mayor frecuencia la volatilidad anualizada ($\sigma$) en lugar de la volatilidad diaria. Teniendo en cuenta que en cada año hay 252 días bursátiles:

$$ \sigma^{2} = \sum_{1}^{252} Var_{diaria}$$
$$ \sigma^{2} = 252 \sigma_{diaria}^{2}$$

Se saca la raíz cuadra a ambos lados para calcular la volatilidad:

$$ \sqrt{\sigma^{2}} = \sqrt{252 \sigma_{diaria}^{2}}$$
$$ \sigma = \sigma_{diaria} \sqrt{252}$$

In [ ]:
# Calculamos la volatilidad diaria con los retornos logaritmicos.
vol_d = np.std(df['Log Returns'])

# Anualizamos la volatilidad diaria.
vol_a = vol_d * np.sqrt(252)

print("Volatilidad diaria: {:.4f} %".format(100*vol_d))
print("Volatilidad anualizada: {:.4f} %".format(100*vol_a))

## 6. Normalidad de los retornos

### 6.1. Histogram

In [ ]:
# Create a histogram with k bins
k = int(math.sqrt(len(df['Log Returns'])))

plt.hist(df['Log Returns'], bins=k)

# Add labels and a title
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.title('Histogram of Log-Returns')

# Show the plot
plt.show()

### 6.2. Jarque-Bera test

In [ ]:
# perform the Jarque-Bera test
jb_value, p_value = jarque_bera(df['Log Returns'])

# print the results
print("Jarque-Bera value: ", jb_value)
print("p-value: ", p_value)

if p_value > 0.05:
    print("The data is normally distributed")
else:
    print("The data is not normally distributed")

In [ ]:
stats.jarque_bera(df['Log Returns'])

## 7. Autocorrelación

### 7.1. Autocorrelation

In [ ]:
# Crear el autocorrelograma
plot_acf(df['Log Returns'], lags = 21)
plt.show()

In [ ]:
df['Log Returns^2'] = df['Log Returns']**2

In [ ]:
# Crear el autocorrelograma de retornos al cuadrado
plot_acf(df['Log Returns^2'], lags = 21)
plt.show()

### 7.2. Partial Autocorrelation

In [ ]:
# Crear el autocorrelograma parcial
plot_pacf(df['Log Returns'], lags = 21)
plt.show()

### 7.3. Ljung-Box test

In [ ]:
def ljung_box(x, lags):
  #   Performs the Ljung-Box test for autocorrelation in a time series.
  #   :param x: the time series data
  #   :param lags: the number of lags to use in the test
  #   :param alpha: the significance level
  #   :return: the test statistic and p-value
  n = len(x)
  Q = n * (n + 2) * np.sum([(np.corrcoef(x[:-i], x[i:])[0, 1])**2 / (n - i) for i in range(1, lags + 1)])
  df = lags
  p_value = 1 - chi2.cdf(Q, df)
  return Q, p_value

In [ ]:
ljung_box(df['Log Returns'],5)

In [ ]:
ljung_box(df['Log Returns^2'],5)

## 8. EWMA Function (Risk Metrics)

In [ ]:
# calculate the rolling standard deviation using the Risk Metrics model
window = 2*252
lambda_param = 0.94
variance = [df['Log Returns'][:window].var()]
for i in range(window, len(df['Log Returns'])):
    variance.append(lambda_param * variance[-1] + (1 - lambda_param) * df['Log Returns'][i-window:i].var())
std = pd.Series(np.sqrt(variance)*np.sqrt(252))

# The std series has one more element than the desired date index.
# The first element of 'std' represents the initial variance over the first 'window' days,
# while the subsequent elements represent the rolling variance from 'window' day onwards.
# To align with 'df.index[window:]', which starts from the 'window'-th day,
# we should drop the first element of 'std'.
std = std.iloc[1:]
std.index = df.index[window:]

# plot the results
import matplotlib.pyplot as plt
plt.plot(std)
plt.title("Risk Metrics Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility")
plt.show()

In [ ]:
std.iloc[-1]

## 9. Implied Volatility

In [ ]:
def black_scholes_call(S, K, r, sigma, T):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    N = norm.cdf
    return S*N(d1) - K*np.exp(-r*T)*N(d2)

def implied_volatility(S, K, r, T, market_price, sigma_guess=0.3):
    """
    Calculate the implied volatility of a European call option using the Black-Scholes formula.

    Parameters:
    S (float): underlying asset price
    K (float): option strike price
    r (float): risk-free interest rate
    T (float): time to maturity in years
    market_price (float): observed market price of the option
    sigma_guess (float, optional): initial guess for the volatility

    Returns:
    float: the implied volatility
    """
    def f(sigma):
        return black_scholes_call(S, K, r, sigma, T) - market_price
    return brentq(f, 0.0001, 1, xtol=1e-10, rtol=1e-10, maxiter=1000)

# Ejemplo

In [ ]:
S = 251.56
K = 290
r = 0.0444
T = 3/12
market_price = 5.75

implied_vol = implied_volatility(S, K, r, T, market_price)
print("Implied volatility: {:.4f}".format(implied_vol))